# 03A — SLICES Domain-Adaptive Pretraining: Mac smoke test

> **DEVELOPMENT / SMOKE TEST ONLY — NOT A FINAL SCIENTIFIC RUN**  
> Este cuaderno valida que el pipeline completo funciona antes de solicitar recursos del cluster. No genera ni publica el modelo final de tesis. El entrenamiento científico completo se ejecuta aparte en MSI mediante una solicitud SLURM explícita.

## Qué verifica

Desde el CSV Alexandria ya convertido a SLICES, este cuaderno carga el tokenizer adaptado, comprueba la gramática, reutiliza el split persistente por composición, prepara secuencias/chunks sin partir tripletas de aristas, inicializa ChemBERTa para MLM, evalúa máscaras fijas antes y después, entrena un epoch de depuración, guarda y vuelve a cargar un checkpoint, y registra un resultado PASS/FAIL.

El código científico compartido vive en `dapt/scripts/dapt_utils.py`; 03A y 03B importan ese mismo módulo. El subset Mac se elige después de asignar cada composición a train/validation/holdout; nunca se reparten chunks al azar.

**Parámetros SLICES/DAPT:** tokenizer `audit/tokenizers/chemberta_zinc_slices_adapted/`; modelo base `seyonec/ChemBERTa-zinc-base-v1`; objetivo MLM; máscara dinámica de entrenamiento 15%; máscaras de diagnóstico fijas con seed 12345. La precisión local se fija de forma conservadora en FP32. La selección del dispositivo consulta MPS y, si no está disponible, usa CPU. La política `mac` deliberadamente no consulta CUDA.


## 1. Imports, versiones y configuración

La configuración ajustable está en `dapt/configs/dapt_mac.yaml`. Por defecto procesa hasta 3,000 estructuras, un epoch, batch de entrenamiento 2 y acumulación de gradiente 8. Cambiar `max_structures` a 1,000, 3,000 o 5,000 no exige modificar el código del notebook.

In [ ]:
from pathlib import Path
import json
import sys
import yaml
import numpy as np
import pandas as pd
import torch
import transformers
import matplotlib.pyplot as plt
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
while not ((PROJECT_ROOT / "dapt").is_dir() and (PROJECT_ROOT / "audit").is_dir()):
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError("Abre Jupyter desde cualquier directorio dentro del proyecto Transformers.")
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "dapt" / "scripts"))
from dapt_utils import (
    create_timestamped_run, detect_device, find_alexandria_csv,
    generate_scientific_plots, run_dapt_pipeline,
)

CONFIG_PATH = PROJECT_ROOT / "dapt/configs/dapt_mac.yaml"
config = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Config:", CONFIG_PATH)
display(pd.Series(config, name="configured value").to_frame())


## 2. Rutas de entrada, salidas y protección del modelo final

El dataset se localiza automáticamente como `alexandria_clean_all.csv` bajo `data/`, salvo que se defina `dataset_path` en YAML. Cada ejecución crea una carpeta temporal nueva con hora local en `dapt/outputs/mac/`. El modelo publicado canónicamente en `dapt/models/chemberta_zinc_slices_dapt/` queda reservado exclusivamente para el runner MSI aceptado; 03A invoca la lógica común con `publish_final_model=False`.

In [ ]:
configured_dataset = config.get("dataset_path")
DATASET_PATH = (PROJECT_ROOT / configured_dataset).resolve() if configured_dataset else find_alexandria_csv(PROJECT_ROOT)
TOKENIZER_PATH = (PROJECT_ROOT / config["tokenizer_path"]).resolve()
SPLIT_PATH = (PROJECT_ROOT / config["split_path"]).resolve()
RUN_DIR = create_timestamped_run(PROJECT_ROOT / "dapt/outputs/mac", "mac_smoke")

print("Alexandria CSV:", DATASET_PATH)
print("Adapted tokenizer:", TOKENIZER_PATH)
print("Persistent composition split:", SPLIT_PATH)
print("This run's outputs:", RUN_DIR)
print("Configured smoke-test limit:", config["max_structures"])
print("Final-model publishing:", False)


## 3. Política local de hardware

En un Mac M4 Pro se espera `mps` cuando la instalación de PyTorch lo ofrece; en caso contrario, el smoke test usa CPU. Aquí no se intenta CUDA y tampoco se activa mixed precision automáticamente: MPS/CPU operan en FP32 para priorizar estabilidad y detectar problemas funcionales.

In [ ]:
device, device_info = detect_device("mac", require_cuda=False)
print("Running device:", device.type.upper())
print("MPS available:", device_info["mps_available"])
print("CUDA queried:", device_info["cuda_probed"])
assert device.type in {"mps", "cpu"}
assert device_info["cuda_probed"] is False


## 4. Smoke test integrado

La función `run_dapt_pipeline` ejecuta en un orden reproducible: lectura sin filtrar filas, auditoría lexical, asignación/reutilización del split global, subset estratificado según etiquetas ya asignadas, parsing, chunking grammar-aware, máscaras fijas, evaluación PRE, resize estándar del vocabulario, entrenamiento MLM, checkpoint/best checkpoint, evaluación POST y comparación por las mismas posiciones enmascaradas.

Las embeddings agregadas se inicializan con el comportamiento estándar de `resize_token_embeddings`; no se fabrican embeddings químicos manualmente. La comparación PRE/POST usa el mismo tokenizer, arquitectura, vocabulario y máscaras.

In [ ]:
config["dataset_path"] = str(DATASET_PATH)
config["tokenizer_path"] = str(TOKENIZER_PATH)
config["split_path"] = str(SPLIT_PATH)
(RUN_DIR / "resolved_config.yaml").write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")

result = run_dapt_pipeline(
    config=config,
    project_root=PROJECT_ROOT,
    run_dir=RUN_DIR,
    device_policy="mac",       # No consulta CUDA; MPS si está disponible, CPU en caso contrario.
    require_cuda=False,
    publish_final_model=False,  # Nunca publicar un modelo final desde el Mac.
    smoke_mode=True,            # No genera los 20 gráficos científicos MSI.
    resume=None,
)
summary = result["summary"]
prepared = result["prepared"]
print("Run finished:", RUN_DIR)
display(pd.DataFrame(prepared["chunking_statistics"]))
display(pd.DataFrame([summary["critical_checks"]]).T.rename(columns={0:"passed"}))


## 5. Construir los checks específicos de 03A y los tres gráficos de diagnóstico

Este bloque exige evidencia de datos, tokenizer, fidelidad, split sin fuga, edges preservadas, embedding original, entrenamiento con pérdidas/normas finitas, evaluaciones PRE/POST y recarga idéntica del checkpoint. Los gráficos locales son breves y no sustituyen la batería científica completa MSI.

In [ ]:
history_path = RUN_DIR / "training_history.csv"
history = pd.read_csv(history_path) if history_path.exists() else pd.DataFrame()
chunk_stats = pd.read_csv(RUN_DIR / "chunking_statistics.csv")
pre_val = json.loads((RUN_DIR / "pre_dapt_validation_metrics.json").read_text())
post_val = json.loads((RUN_DIR / "post_dapt_validation_metrics.json").read_text())
pre_hold = json.loads((RUN_DIR / "pre_dapt_holdout_metrics.json").read_text())
post_hold = json.loads((RUN_DIR / "post_dapt_holdout_metrics.json").read_text())

critical = summary["critical_checks"]
loss_columns = [c for c in ["train_loss", "eval_loss"] if c in history]
loss_values = history[loss_columns].to_numpy(dtype=float) if loss_columns else np.array([])
finite_losses = bool(loss_values.size and np.isfinite(loss_values[~np.isnan(loss_values)]).all())
gradient_values = pd.to_numeric(history.get("gradient_norm", pd.Series(dtype=float)), errors="coerce").dropna()
finite_gradients = bool(len(gradient_values) and np.isfinite(gradient_values).all())
checks = {
    "dataset_loaded": len(prepared["structures"]) > 0,
    "tokenizer_loaded": len(result["tokenizer"]) > 0,
    "grammar_fidelity_passed": summary["fidelity_rate"] >= float(config["minimum_grammar_fidelity"]),
    "composition_split_clean": bool(critical["composition_split_clean"]),
    "chunking_completed": len(prepared["chunks"]) > 0,
    "all_edges_preserved": float(chunk_stats["fraction_edges_preserved_all_selected"].iloc[0]) == 1.0,
    "model_loaded_and_resized": bool(critical["tokenizer_embedding_size_match"]),
    "original_embeddings_preserved": bool(critical["original_embeddings_preserved"]),
    "training_completed": int(summary["epochs_completed"]) >= 1,
    "finite_losses": finite_losses,
    "finite_gradients": finite_gradients,
    "checkpoint_written": (RUN_DIR / "best_checkpoint").is_dir() and (RUN_DIR / "checkpoints").is_dir(),
    "checkpoint_reload_passed": bool(critical["checkpoint_reload_passed"]),
    "pre_dapt_evaluation_completed": pre_val.get("masked_count", 0) > 0 and pre_hold.get("masked_count", 0) > 0,
    "post_dapt_evaluation_completed": post_val.get("masked_count", 0) > 0 and post_hold.get("masked_count", 0) > 0,
}
checks["pipeline_passed"] = all(checks.values())
checks["ready_for_msi"] = checks["pipeline_passed"]

# Compact diagnostic figure 1: training and fixed-mask validation loss.
fig, ax = plt.subplots(figsize=(8, 4.5))
if not history.empty:
    train_rows = history.dropna(subset=["train_loss"])
    eval_rows = history.dropna(subset=["eval_loss"])
    if len(train_rows): ax.plot(train_rows["step"], train_rows["train_loss"], marker=".", label="Training")
    if len(eval_rows): ax.plot(eval_rows["step"], eval_rows["eval_loss"], marker="o", label="Validation fixed masks")
ax.set(title="Mac smoke test — MLM loss", xlabel="Optimizer step", ylabel="Cross-entropy")
ax.grid(alpha=.25); ax.legend(); fig.tight_layout(); fig.savefig(RUN_DIR / "01_mac_training_curve.png", dpi=160); plt.show(); plt.close(fig)

# Diagnostic figure 2: same tokenizer and fixed masks, before versus after this smoke run.
fig, ax = plt.subplots(figsize=(7, 4.5))
metrics = ["loss", "top1_accuracy", "top5_accuracy"]
x = np.arange(len(metrics)); width = .36
ax.bar(x-width/2, [pre_val.get(m, np.nan) for m in metrics], width, label="PRE")
ax.bar(x+width/2, [post_val.get(m, np.nan) for m in metrics], width, label="POST")
ax.set_xticks(x, metrics); ax.set_title("Validation diagnostics (smoke test only)"); ax.legend(); ax.grid(axis="y", alpha=.25)
fig.tight_layout(); fig.savefig(RUN_DIR / "02_mac_pre_vs_post.png", dpi=160); plt.show(); plt.close(fig)

# Diagnostic figure 3: a few real fixed-mask examples with PRE/POST predictions.
examples_path = RUN_DIR / "masked_prediction_examples.csv"
examples = pd.read_csv(examples_path) if examples_path.exists() else pd.DataFrame()
fig, ax = plt.subplots(figsize=(12, 5)); ax.axis("off")
if not examples.empty:
    lines = []
    for _, row in examples.head(3).iterrows():
        lines.append(f"{row.get('token_category_pre','?')} | true={row.get('true_token_pre','?')}\\nPRE top-5: {row.get('top5_tokens_json_pre','')}\\nPOST top-5: {row.get('top5_tokens_json_post','')}")
    ax.text(.01, .98, "\\n\\n".join(lines), va="top", family="monospace", fontsize=9, wrap=True)
else:
    ax.text(.02, .95, "No representative masked examples were available.", va="top")
ax.set_title("Fixed-mask prediction examples — diagnostic, not a scientific conclusion")
fig.tight_layout(); fig.savefig(RUN_DIR / "03_mac_prediction_examples.png", dpi=160); plt.show(); plt.close(fig)

status = {
    "pipeline_passed": checks["pipeline_passed"],
    "grammar_fidelity_passed": checks["grammar_fidelity_passed"],
    "chunking_passed": checks["chunking_completed"] and checks["all_edges_preserved"],
    "training_completed": checks["training_completed"],
    "finite_losses": checks["finite_losses"],
    "finite_gradients": checks["finite_gradients"],
    "checkpoint_reload_passed": checks["checkpoint_reload_passed"],
    "ready_for_msi": checks["ready_for_msi"],
    "checks": checks,
    "run_dir": str(RUN_DIR),
    "scientific_run": False,
}
(RUN_DIR / "mac_smoke_test_status.json").write_text(json.dumps(status, indent=2), encoding="utf-8")
summary["mac_smoke_checks"] = checks
summary["ready_for_msi"] = status["ready_for_msi"]
(RUN_DIR / "mac_smoke_summary.json").write_text(json.dumps(summary, indent=2, default=str), encoding="utf-8")
display(pd.Series(checks, name="PASS").to_frame())


## 6. Auditoría rápida del smoke test

Las métricas ayudan a detectar fallos de ejecución, pero un único epoch con subset pequeño no demuestra adaptación científica. No interpretar una pérdida PRE/POST local como resultado de tesis; no mover estos pesos a `dapt/models/chemberta_zinc_slices_dapt/`.

In [ ]:
print("=" * 48)
print("MAC DAPT SMOKE TEST")
print("=" * 48)
print("Pipeline status:", "PASS" if status["pipeline_passed"] else "FAIL")
print("This run is NOT a scientific result.")
print("READY FOR MSI:", "YES" if status["ready_for_msi"] else "NO")
print("Run artifacts:", RUN_DIR)
print("=" * 48)
assert len(prepared["structures"]) <= int(config["max_structures"])
assert SPLIT_PATH.is_file(), "The one-time global composition split should persist for MSI."


## 7. Qué sigue (ejecución separada)

Si `READY FOR MSI: YES`, sincroniza el proyecto con el cluster y revisa partición, cuenta, formato de GPU, entorno Conda y límite de tiempo con MSI. El notebook local **no** lanza jobs. En el cluster, desde la raíz del proyecto, la solicitud futura será explícita: `sbatch dapt/scripts/submit_dapt_msi.slurm`. El script necesita `CONDA_ENV_NAME` definido y no inventa una cuenta privada.